# Example: Surface brightness profile extraction and fitting

This thread shows how to read data, extract surface brightness profiles, fit data and extract density profiles with _PyProffit_. 

## Reading data ##

We start by loading the packages:

In [ ]:
import numpy as np
import pyproffit
import matplotlib.pyplot as plt
import os

# Change this to the proper directory containing your run
os.chdir('../../validation/') 

Now we load the data inside a [Data](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.data.Data) object in PyProffit structure:

In [ ]:
dat=pyproffit.Data(imglink='b_37.fits.gz',explink='expose_mask_37.fits.gz',
                   bkglink='back_37.fits.gz')

1. Here _imglink=_'b_37.fits.gz'_ is the link to the image file (count map) to be loaded.
2. _explink=_'expose_mask_37.fits.gz' is the exposure map.   
3. _bkglink=_'back_37.fits.gz' is the background map.

The images are then loaded into the [Data](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.data.Data) structure and can be easily accessed as below:

In [ ]:
fig = plt.figure(figsize=(20,20))
s1=plt.subplot(221)
plt.imshow(dat.img,aspect='auto')
s2=plt.subplot(222)
plt.imshow(dat.exposure,aspect='auto')

All the areas with zero exposure will be automatically excluded. We can ignore additional regions using the [region](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.data.Data.region) method of the [Data](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.data.Data) class, which loads a DS9 region file (in image or FK5 format):

In [ ]:
dat.region('test_region.reg')

The [Data](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.data.Data) structure also contains the [dmfilth](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.data.Data.dmfilth) method, which can be used to fill the masked areas. The method computes a 2D spline interpolation in between the gaps and generates a Poisson realization of the spline interpolated data, such that the filled holes have similar statistical properties to their surroundings

In [ ]:
dat.dmfilth()

## Profile extraction ##

Now we define a [Profile](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.profextract.Profile) object in the following way:


In [ ]:
prof=pyproffit.Profile(dat,center_choice='centroid',maxrad=45.,binsize=20.,centroid_region=30.)

<h3>Profile class options</h3>

The class [Profile](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.profextract.Profile) is designed to contain all the Proffit profile extraction features (not all of them have been implemented yet). The "center_choice" argument specifies the choice of the center:

- center_choice='centroid': compute image centroid and ellipticity 

The other arguments are the following:

* maxrad: define the maximum radius of the profile (in arcmin)
* binsize: the width of the bins (in arcsec)
* binsize: minimum bin size in arcsec 
* centroid_region: for centroid calculation (center_choice='centroid'), optionally provide a radius within which the centroid will be computed, instead of the entire image.

<h3> Now let's extract the profile...</h3>

In [ ]:
prof.SBprofile(ellipse_ratio=prof.ellratio,rotation_angle=prof.ellangle+180.)

<p>Here we have extracted a profile in elliptical annuli centered on the image centroid (see above), with an ellipse axis ratio (major/minor) and position angle calculated with principal component analysis. If ellipse_ratio and ellipse_angle are left blank circular annuli are used.</p>

- ellipse_ratio: the ratio of major to minor axis (a/b) of the ellipse (default=1, i.e. circular annuli)
- rotation_angle: rotation angle of the ellipse from the R.A. axis (default=0)

<p> Now let's plot the profile...</p>

In [ ]:
prof.Plot()

## Defining a model ##

Models can be defined using the [Model](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.models.Model) class.

In [ ]:
mod=pyproffit.Model(pyproffit.BetaModel)

To check the parameters of the [BetaModel](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.models.BetaModel) function,

## Fitting the data ##

To fit the extracted profiles PyProffit provides the [Fitter](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.fitting.Fitter) class, which takes a [Profile](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.profextract.Profile) and a [Model](https://pyproffit.readthedocs.io/en/latest/pyproffit.html#pyproffit.models.Model) object as input:

In [ ]:
fitobj=pyproffit.Fitter(model=mod, profile=prof, beta=0.7, rc=2.,norm=-2,bkg=-4)
fitobj.Migrad()

The fit provides the best-fit parameters with their errors and the covariance matrix. The fit statistic is provided at the bottom.

In [ ]:
prof.Plot(model=mod)